# Dashboard — FormalizaEnte

Este notebook é a camada de consumo da PoC. Ele lê somente a **Gold** gerada pelo dbt e transforma os resultados em indicadores, gráficos e uma narrativa para apresentação.

**Pergunta de negócio**

> Qual é a taxa de cobertura das solicitações pelo FormalizaEnte, em qual esfera — Município, Estado ou União — elas são resolvidas e como essa cobertura varia por período, categoria e área de origem?

Antes de abrir o dashboard, execute na raiz do projeto:

```bash
python -m src.pipeline
dbt build
```

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
def encontrar_raiz_projeto(inicio: Path) -> Path:
    atual = inicio.resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dbt_project.yml").exists():
            return candidato
    raise FileNotFoundError("Abra o notebook dentro do repositório FormalizaEnte.")

PROJECT_ROOT = encontrar_raiz_projeto(Path.cwd())
WAREHOUSE = PROJECT_ROOT / "warehouse" / "formalizaente.duckdb"

if not WAREHOUSE.exists():
    raise FileNotFoundError(
        f"Warehouse não encontrado em {WAREHOUSE}. "
        "Execute `python -m src.pipeline` e `dbt build` na raiz do projeto."
    )

con = duckdb.connect(str(WAREHOUSE), read_only=True)
print(f"Projeto: {PROJECT_ROOT}")
print(f"Warehouse: {WAREHOUSE}")

## 1. Checagem das camadas físicas

O projeto segue o padrão Medallion: `raw → bronze → silver → gold`. Esta célula mostra se as pastas existem e quantos arquivos derivados foram gerados.

In [ ]:
camadas = []
for nome in ["raw", "bronze", "silver", "gold"]:
    caminho = PROJECT_ROOT / "data" / nome
    arquivos = [p for p in caminho.rglob("*") if p.is_file() and p.name != ".gitkeep"]
    camadas.append({"camada": nome.upper(), "existe": caminho.exists(), "arquivos": len(arquivos)})

camadas_df = pd.DataFrame(camadas)
display(camadas_df)

fig = px.bar(
    camadas_df,
    x="camada",
    y="arquivos",
    text="arquivos",
    title="Arquivos por camada física do pipeline",
)
fig.update_traces(textposition="outside")
fig.update_yaxes(title="Quantidade de arquivos")
fig.update_xaxes(title="Camada")
fig.show()

## 2. Indicadores principais

Os números vêm de `gold.agg_cobertura_geral`. O notebook não recalcula a regra de negócio.

In [ ]:
kpis = con.execute(
    """
    select
        total_solicitacoes_validas,
        solicitacoes_atendidas,
        solicitacoes_nao_encontradas,
        taxa_cobertura_pct,
        solicitacoes_com_ambiguidade,
        taxa_ambiguidade_pct
    from gold.agg_cobertura_geral
    """
).df()

linha = kpis.iloc[0]

cards = [
    ("Solicitações válidas", int(linha["total_solicitacoes_validas"])),
    ("Cobertas", int(linha["solicitacoes_atendidas"])),
    ("Não encontradas", int(linha["solicitacoes_nao_encontradas"])),
    ("Cobertura", f'{linha["taxa_cobertura_pct"]:.2f}%'),
    ("Ambíguas", int(linha["solicitacoes_com_ambiguidade"])),
    ("Ambiguidade", f'{linha["taxa_ambiguidade_pct"]:.2f}%'),
]

html = '<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin:10px 0 25px 0;">'
for titulo, valor in cards:
    html += (
        '<div style="border:1px solid #d0d7de;border-radius:12px;padding:18px;background:#fff;">'
        f'<div style="font-size:13px;color:#57606a;margin-bottom:8px;">{titulo}</div>'
        f'<div style="font-size:30px;font-weight:700;color:#0b1f33;">{valor}</div>'
        '</div>'
    )
html += '</div>'
display(HTML(html))

kpis

## 3. Cobertura x não encontrado

Visão executiva da pergunta principal: quantas solicitações válidas encontraram correspondência em alguma lista normativa.

In [ ]:
funil = pd.DataFrame([
    {"status": "Cobertas", "total": int(linha["solicitacoes_atendidas"])},
    {"status": "Não encontradas", "total": int(linha["solicitacoes_nao_encontradas"])},
])

fig = px.pie(
    funil,
    names="status",
    values="total",
    hole=0.55,
    title="Cobertura geral das solicitações válidas",
)
fig.update_traces(textposition="inside", textinfo="label+percent+value")
fig.show()

## 4. Distribuição por esfera

A esfera já foi definida pelo dbt com base na prioridade versionada: REMUME → Município, RESME → Estado, RENAME → União.

In [ ]:
esfera = con.execute(
    """
    select
        esfera_resolucao,
        total_solicitacoes,
        percentual_solicitacoes
    from gold.agg_distribuicao_esfera
    order by total_solicitacoes desc, esfera_resolucao
    """
).df()

display(esfera)

fig = px.bar(
    esfera,
    x="esfera_resolucao",
    y="total_solicitacoes",
    text="percentual_solicitacoes",
    title="Distribuição das solicitações por esfera de resolução",
)
fig.update_traces(texttemplate="%{text:.2f}%")
fig.update_yaxes(title="Solicitações")
fig.update_xaxes(title="Esfera")
fig.show()

## 5. Evolução mensal

Mostra se a cobertura muda ao longo do período analisado.

In [ ]:
mensal = con.execute(
    """
    select
        mes_referencia,
        total_solicitacoes,
        solicitacoes_atendidas,
        taxa_cobertura_pct
    from gold.agg_cobertura_mensal
    order by mes_referencia
    """
).df()

display(mensal)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=mensal["mes_referencia"],
    y=mensal["total_solicitacoes"],
    name="Solicitações válidas",
    yaxis="y2",
    opacity=0.35,
))
fig.add_trace(go.Scatter(
    x=mensal["mes_referencia"],
    y=mensal["taxa_cobertura_pct"],
    mode="lines+markers+text",
    name="Cobertura (%)",
    text=[f"{v:.2f}%" for v in mensal["taxa_cobertura_pct"]],
    textposition="top center",
))
fig.update_layout(
    title="Cobertura mensal e volume de solicitações",
    xaxis_title="Mês",
    yaxis=dict(title="Cobertura (%)", range=[0, 110]),
    yaxis2=dict(title="Solicitações", overlaying="y", side="right", showgrid=False),
    legend=dict(orientation="h"),
)
fig.show()

## 6. Cobertura por área de origem

In [ ]:
area = con.execute(
    """
    select
        area_origem,
        total_solicitacoes,
        solicitacoes_atendidas,
        taxa_cobertura_pct
    from gold.agg_cobertura_area
    order by taxa_cobertura_pct desc, area_origem
    """
).df()

display(area)

fig = px.bar(
    area,
    x="area_origem",
    y="taxa_cobertura_pct",
    text="taxa_cobertura_pct",
    hover_data=["total_solicitacoes", "solicitacoes_atendidas"],
    title="Taxa de cobertura por área de origem",
)
fig.update_traces(texttemplate="%{text:.2f}%")
fig.update_yaxes(title="Cobertura (%)", range=[0, 100])
fig.update_xaxes(title="Área")
fig.show()

## 7. Cobertura por categoria

In [ ]:
categoria = con.execute(
    """
    select
        categoria,
        total_solicitacoes,
        solicitacoes_atendidas,
        taxa_cobertura_pct
    from gold.agg_cobertura_categoria
    order by taxa_cobertura_pct, categoria
    """
).df()

display(categoria)

fig = px.bar(
    categoria,
    x="taxa_cobertura_pct",
    y="categoria",
    orientation="h",
    text="taxa_cobertura_pct",
    hover_data=["total_solicitacoes", "solicitacoes_atendidas"],
    title="Taxa de cobertura por categoria",
)
fig.update_traces(texttemplate="%{text:.2f}%")
fig.update_xaxes(title="Cobertura (%)", range=[0, 100])
fig.update_yaxes(title="Categoria")
fig.show()

## 8. Matriz área × esfera

Exploração adicional sobre a fato Gold. A classificação já vem pronta de `gold.fct_solicitacoes`.

In [ ]:
matriz = con.execute(
    """
    select
        area_origem,
        esfera_resolucao,
        count(*) as total_solicitacoes
    from gold.fct_solicitacoes
    group by 1, 2
    order by 1, 2
    """
).df()

pivot = matriz.pivot_table(
    index="area_origem",
    columns="esfera_resolucao",
    values="total_solicitacoes",
    fill_value=0,
)
display(pivot)

fig = px.density_heatmap(
    matriz,
    x="esfera_resolucao",
    y="area_origem",
    z="total_solicitacoes",
    text_auto=True,
    title="Matriz de solicitações por área e esfera",
)
fig.update_xaxes(title="Esfera")
fig.update_yaxes(title="Área")
fig.show()

## 9. Ambiguidade por solicitação

Uma solicitação é ambígua quando o mesmo item aparece em duas ou mais listas. O dashboard mostra essa informação, mas a regra foi calculada antes, no dbt.

In [ ]:
ambiguidade = con.execute(
    """
    select
        indicador_ambiguidade,
        count(*) as total
    from gold.fct_solicitacoes
    group by 1
    order by 1
    """
).df()
ambiguidade["status"] = ambiguidade["indicador_ambiguidade"].map({0: "Sem ambiguidade", 1: "Ambígua"})

display(ambiguidade[["status", "total"]])

fig = px.pie(
    ambiguidade,
    names="status",
    values="total",
    hole=0.45,
    title="Participação de solicitações ambíguas",
)
fig.update_traces(textinfo="label+percent+value")
fig.show()

## 10. Tabela final de resposta

Esta tabela é o mesmo mart usado pela consulta oficial `queries/resposta_negocio.sql`.

In [ ]:
resposta = con.execute(
    """
    select
        tipo_recorte,
        recorte,
        total_solicitacoes,
        solicitacoes_atendidas,
        taxa_cobertura_pct,
        taxa_ambiguidade_pct,
        percentual_distribuicao_pct
    from gold.agg_resposta_gerencial
    order by ordem_exibicao, recorte
    """
).df()

display(resposta)

## 11. Destaques para falar na apresentação

In [ ]:
melhor_area = area.sort_values(
    ["taxa_cobertura_pct", "area_origem"],
    ascending=[False, True],
).iloc[0]

menor_area = area.sort_values(
    ["taxa_cobertura_pct", "area_origem"],
    ascending=[True, True],
).iloc[0]

melhor_mes = mensal.sort_values(
    ["taxa_cobertura_pct", "mes_referencia"],
    ascending=[False, True],
).iloc[0]

print(
    f'Cobertura geral: {linha["taxa_cobertura_pct"]:.2f}% '
    f'({int(linha["solicitacoes_atendidas"])}/'
    f'{int(linha["total_solicitacoes_validas"])} solicitações válidas).'
)
print(
    f'Ambiguidade: {linha["taxa_ambiguidade_pct"]:.2f}% '
    f'({int(linha["solicitacoes_com_ambiguidade"])} solicitações).'
)
print(
    f'Maior cobertura por área: {melhor_area["area_origem"]} '
    f'({melhor_area["taxa_cobertura_pct"]:.2f}%).'
)
print(
    f'Menor cobertura por área: {menor_area["area_origem"]} '
    f'({menor_area["taxa_cobertura_pct"]:.2f}%).'
)
print(
    f'Maior cobertura mensal: {melhor_mes["mes_referencia"]} '
    f'({melhor_mes["taxa_cobertura_pct"]:.2f}%).'
)

## Encerramento

O notebook demonstra consumo analítico: ele não corrige dado, não deduplica, não aplica prioridade e não acessa fontes brutas. A lógica foi resolvida nas camadas anteriores.

In [ ]:
con.close()
print("Conexão encerrada.")